# Tutorial 2: Working with Language Models in LangChain

In this tutorial, we'll explore how to work with language models in LangChain, focusing on the Groq LLM. We'll cover connecting to the model, creating prompt templates, building chains, and handling responses.

## 1. Connecting to Language Models

First, let's set up our environment and connect to the Groq LLM:

In [1]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

load_dotenv()

# Initialize the Groq LLM
llm = ChatGroq(
        model_name="llama-3.1-8b-instant",
        temperature=0.1,
        model_kwargs={"top_p": 0.2, "seed": 1337}
    )

# Test the connection
response = llm.invoke("Hello, world!")
print(response)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


content='Hello, world. What can I help you with today?' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 39, 'total_tokens': 52, 'completion_time': 0.02310926, 'completion_tokens_details': None, 'prompt_time': 0.00262943, 'prompt_tokens_details': None, 'queue_time': 0.048292222, 'total_time': 0.02573869}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e09ee421cf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019dfd24-395c-7cb2-85a7-266a4fe41cad-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 39, 'output_tokens': 13, 'total_tokens': 52}


## 2. Creating Prompt Templates

Prompt templates allow us to create reusable prompts with input variables:

In [5]:
# Define a simple prompt template
template = """Answer the following question:
You answers in the {language} language.
Question: {question}
Answer: Let's approach this step-by-step:"""


prompt = PromptTemplate(template=template, input_variables=["question","language"])

# Use the prompt template
question = "What is the capital of France?"
formatted_prompt = prompt.format(question=question,language="English")
print(formatted_prompt)

Answer the following question:
You answers in the English language.
Question: What is the capital of France?
Answer: Let's approach this step-by-step:


## 3. Building Simple Prompt Chains

Now, let's create a chain that combines our prompt template with the language model:

In [6]:
# Create a chain
chain = prompt | llm

# Run the chain
result = chain.invoke({"question":"What is the speed of light?","language":"French"})
print(result.content)

La vitesse de la lumière. C'est une constante fondamentale de l'univers. 

Pour la déterminer, nous devons nous référer à la définition de la vitesse. La vitesse est le rapport entre la distance parcourue et le temps mis pour parcourir cette distance.

La vitesse de la lumière est généralement notée par c. Elle est égale à environ 299 792 458 mètres par seconde. C'est une valeur très précise qui a été déterminée grâce à des mesures extrêmement précises.

Il est important de noter que la vitesse de la lumière est constante dans le vide et ne dépend pas de la source lumineuse ou de l'observateur. C'est une propriété fondamentale de la lumière qui a été confirmée par de nombreuses expériences et observations.


## 4. Handling Model Responses

Let's explore different ways to handle and process model responses:

In [7]:
# Get the raw response
raw_response = llm.invoke("List three prime numbers.")
print("Raw response:", raw_response)

# Using the chain with a dictionary input
chain_response = chain.invoke({"question": "tree Names","language": "French"})
print("\nChain response:", chain_response.content)

Raw response: content='Here are three prime numbers:\n\n1. 23\n2. 37\n3. 53' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 40, 'total_tokens': 61, 'completion_time': 0.027399063, 'completion_tokens_details': None, 'prompt_time': 0.002991876, 'prompt_tokens_details': None, 'queue_time': 0.048792776, 'total_time': 0.030390939}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d9492c3c54', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019dfd24-abbc-7ad2-8e90-3a88d8e283a1-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 40, 'output_tokens': 21, 'total_tokens': 61}

Chain response: Les noms de quelques arbres sont :

1. L'arbre de Noël (le sapin)
2. L'arbre à pain (le chêne)
3. L'arbre à miel (le châtaignier)
4. L'arbre à papier (le papierier)
5. L'arbre à huile (l'olivier)
6. L'arbre à bois (le pin)
7. L'arbre à charpente (le chêne)
8. L'a

In [8]:
# Parsing structured output
from langchain_core.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()
format_instructions = output_parser.get_format_instructions()

list_prompt = PromptTemplate(
    template="List 100 {item}. {format_instructions} write only the colors, nothing else",
    input_variables=["item"],
    partial_variables={"format_instructions": format_instructions}
)

chain = list_prompt | llm |output_parser
result = chain.invoke({"item":"colors"})
print(type(result))
print("\nParsed list:", result)

<class 'list'>

Parsed list: ['Aqua', 'Black', 'Blue', 'Brown', 'Beige', 'Burgundy', 'Coral', 'Cream', 'Cyan', 'Dark Gray', 'Emerald', 'Forest Green', 'Gold', 'Gray', 'Green', 'Hunter Green', 'Indigo', 'Ivory', 'Jade', 'Khaki', 'Lavender', 'Lilac', 'Magenta', 'Maroon', 'Mint', 'Navy Blue', 'Olive', 'Onyx', 'Orange', 'Peach', 'Periwinkle', 'Pink', 'Plum', 'Powder Blue', 'Purple', 'Red', 'Ruby', 'Sage', 'Silver', 'Sky Blue', 'Teal', 'Turquoise', 'Violet', 'White', 'Yellow', 'Charcoal', 'Chocolate', 'Fuchsia', 'Garnet', 'Gray Blue', 'Hunter', 'Jade Green', 'Lavender Gray', 'Mahogany', 'Mauve', 'Mocha', 'Mustard', 'Navy', 'Ochre', 'Pale Blue', 'Pale Green', 'Pale Pink', 'Pale Yellow', 'Periwinkle Blue', 'Plum Brown', 'Powder Pink', 'Red Brown', 'Red Orange', 'Red Violet', "Robin's Egg Blue", 'Royal Blue', 'Sage Green', 'Sea Green', 'Sepia', 'Sienna', 'Silver Gray', 'Sky Blue Green', 'Steel Gray', 'Taupe', 'Tawny', 'Terra Cotta', 'Thistle', 'Tomato', 'Turquoise Blue', 'Violet Blue', 'Violet

## 5. Best Practices for Prompt Engineering

Here are some best practices for effective prompt engineering:

In [9]:
# 1. Be specific and provide context
specific_prompt = PromptTemplate(
    template="You are an expert in {field}. Explain {concept} in simple terms for a beginner.",
    input_variables=["field", "concept"]
)

# 2. Use examples (few-shot learning)
few_shot_prompt = PromptTemplate(
    template="""Classify the sentiment of the following text as positive, negative, or neutral.

Example 1:
Text: I love this product!
Sentiment: Positive

Example 2:
Text: This is the worst experience ever.
Sentiment: Negative

Example 3:
Text: The weather is cloudy today.
Sentiment: Neutral

Now, classify the following text:
Text: {text}
Sentiment:""",
    input_variables=["text"]
)

# 3. Break complex tasks into steps
step_prompt = PromptTemplate(
    template="""To solve the problem '{problem}', follow these steps:
1. Identify the key information
2. Determine the appropriate formula or method
3. Apply the formula or method step-by-step
4. Check your answer

Now, solve the problem:""",
    input_variables=["problem"]
)

# Test the prompts
chains = {
    "Specific": specific_prompt | llm,
    "Few-shot": few_shot_prompt | llm,
    "Step-by-step": step_prompt | llm 
    }

for name, chain in chains.items():
    print(f"\n---------------------------------------------------\n{name} Prompt Result:")
    if name == "Specific":
        print(chain.invoke({"field":"physics", "concept":"quantum entanglement"}).content)
    elif name == "Few-shot":
        print(chain.invoke({"text":"This movie was okay, I guess."}).content)
    else:
        print(chain.invoke({"problem":"Calculate the area of a circle with radius 5 cm"}).content)


---------------------------------------------------
Specific Prompt Result:
Quantum entanglement is a fascinating concept in physics that can be a bit mind-bending, but I'll try to explain it in simple terms.

**What is Quantum Entanglement?**

Imagine you have two toy boxes, one in New York and one in Los Angeles. Inside each box, you have a small ball that can be either red or blue. Now, let's say you have a special connection between the two boxes, so that when you open the box in New York and find out the color of the ball, you instantly know the color of the ball in the box in Los Angeles, even if it's on the other side of the country.

**But here's the weird part:**

When you open the box in New York and find out the color of the ball, the ball in the box in Los Angeles doesn't just suddenly change color to match the one in New York. Instead, the color of the ball in Los Angeles was already determined, even before you opened the box in New York. It's as if the two balls are "con

## Conclusion

In this tutorial, we've explored various aspects of working with language models in LangChain, including connecting to models, creating prompt templates, building chains, handling responses, and implementing best practices for prompt engineering. These skills will serve as a foundation for building more complex applications with LangChain in future tutorials.